In [16]:
from core.file_manager import preprocess_file_manager
from core.visualization_lib import folder_shower, normalize_volume
from core.helper import  copy_NiFty, patients_transform
from core.helper import transform_step_list_to_dictioanry

from core.transformers.nifti_to_raw_transformer import nifti_to_raw_transformer
from core.transformers.anatomy_fill_transformer import anatomy_fill_transformer
from core.transformers.crop_transformer import non_weighted_crop_transformer
from core.transformers.resample_transformer import resample_transformer
from settings.main_settings import test_settings


In [ ]:
original_data_folder =  '/home/robakp/Exeriments1/prostate_lesion_detection/rjozwiak-MGR_dataset_correct/MGR_dataset_correct'

channels = {
    'adc' : 'adc',
    'anatomy' : 'anatomy',
    'dwi' : 'dwi',
    't2' : 't2'
}

target = 'lesion'

file_extention = '.nii.gz'

preprocessing_steps_list = [
    ('start', 'nifty'),
    ('resampling','resampled'),
    ('nifti_to_raw', 'raw'),
    ('filling_anatomy_gaps', 'anatomy_gap_filled'),
    ('cropping', 'cropped'),

]

step_functions = {}

preprocessed_steps = transform_step_list_to_dictioanry(preprocessing_steps_list)

crop_size = (160,160,24)
target_spacing=(0.8, 0.8, 3.5)



file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

In [18]:
copy_NiFty(original_data_folder, file_manager,channels, filter=['3322','001','003','004'], step=preprocessed_steps[preprocessing_steps_list[1][0]]['start'])

step_functions['start'] = lambda: copy_NiFty(
    original_data_folder,
    file_manager,
    channels,
    filter=['3322','001','003','004'],
    step=preprocessed_steps[preprocessing_steps_list[1][0]]['start']
)

In [ ]:

if preprocessed_steps.get('resampling'):
    start_step = preprocessed_steps['resampling']['start']
    end_step = preprocessed_steps['resampling']['end']
    resample_transformer = resample_transformer(target_spacing=target_spacing,crop_size=crop_size, is_label=False)

    step_functions['resampling'] = lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, resample_transformer)

In [ ]:
if preprocessed_steps.get('nifti_to_raw'):
    start_step = preprocessed_steps['nifti_to_raw']['start']
    end_step = preprocessed_steps['nifti_to_raw']['end']
    to_raw_transform = nifti_to_raw_transformer()

    step_functions['nifti_to_raw'] =  lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, to_raw_transform)

In [21]:
if preprocessed_steps.get('filling_anatomy_gaps'):
    start_step = preprocessed_steps['filling_anatomy_gaps']['start']
    end_step = preprocessed_steps['filling_anatomy_gaps']['end']
    gap_fill_transformer = anatomy_fill_transformer()
    
    step_functions['filling_anatomy_gaps'] = lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, gap_fill_transformer)

In [ ]:
if preprocessed_steps.get('cropping'):
    start_step = preprocessed_steps['cropping']['start']
    end_step = preprocessed_steps['cropping']['end']
    nw_crop_transformer = non_weighted_crop_transformer(crop_size)

    step_functions['cropping'] = lambda start=start_step, end=end_step: patients_transform(file_manager, start, end, nw_crop_transformer)

In [24]:
for step, _ in preprocessing_steps_list:
    print(f"Starting step: {step}")
    step_functions[step]()
    print(f"Finished step: {step}")

Starting step: start
Finished step: start
Starting step: resampling
Finished step: resampling
Starting step: nifti_to_raw
Finished step: nifti_to_raw
Starting step: cropping
Finished step: cropping
